# Train on Google Colab (free T4 GPU)

This notebook reproduces the full run on a free Colab GPU: data → cache → 40-epoch
training (~25 min) → post-processing check → prediction figure. The checkpoint
is saved to your Google Drive, so a disconnect doesn't lose it.

**Before you start:** `Runtime → Change runtime type → T4 GPU`.

You need `stage1_train.zip` from the
[Kaggle competition page](https://www.kaggle.com/c/data-science-bowl-2018/data).
Either put it in Google Drive at `MyDrive/nuclei/stage1_train.zip` (recommended:
it survives runtime resets), or upload it when cell 3 asks.

## 1. Check the GPU

In [ ]:
# nvidia-smi lists the attached GPU; if this fails, the runtime has no GPU
# (Runtime -> Change runtime type -> T4 GPU, then rerun).
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Get the code and install dependencies

In [ ]:
# Clone the repo and work from its root: every script imports `src.*`
# relative to the project root.
!git clone -q https://github.com/lina-dh/nuclei-segmentation-dsb2018.git
%cd nuclei-segmentation-dsb2018
# Colab already ships torch, numpy, opencv, scikit-image and matplotlib.
# Only the segmentation stack is added; versions match the original run.
!pip install -q segmentation-models-pytorch==0.5.0 albumentations==2.0.8 timm wandb

## 3. Get the dataset

In [ ]:
import os, shutil
from google.colab import drive

# Mount Drive: the zip is read from it and the checkpoint is saved back to it.
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/nuclei'
os.makedirs(DRIVE_DIR, exist_ok=True)
zip_path = f'{DRIVE_DIR}/stage1_train.zip'

if not os.path.exists(zip_path):
    # Fallback: browser upload (~80 MB). The file is then copied to Drive so
    # the next session can skip this step.
    from google.colab import files
    print('stage1_train.zip not found in Drive - choose it from your computer:')
    up = files.upload()
    name = next(iter(up))
    shutil.copy(name, zip_path)

# The zip holds one folder per sample at its root -> extract into data/stage1_train.
!mkdir -p data/stage1_train
!unzip -q -o "{zip_path}" -d data/stage1_train
!ls data/stage1_train | wc -l   # expect 670

## 4. Build the cache (~3-5 min)
Decodes every image and mask **once** into `.npy` arrays. The train/val split
comes from the committed `.project_state/manifest.json`, so this run uses exactly
the same 534/135 split as the reported numbers.

In [ ]:
!python -m scripts.build_cache

## 5. Train (~25 min)
Same settings as the reported run. GPU math isn't bit-deterministic, so expect
a best val mAP close to, but not exactly, 0.574 (untuned thresholds).

In [ ]:
# --wandb-mode disabled: no account needed. Output goes to runs/colab40/.
!python train.py --epochs 40 --crops-per-epoch 3000 --batch-size 16 \
    --wandb-mode disabled --wandb-run colab40

# Copy the checkpoint to Drive right away - Colab runtimes are temporary.
!cp runs/colab40/best.pt "{DRIVE_DIR}/best.pt" && ls -lh "{DRIVE_DIR}/best.pt"

## 6. Post-processing check with flip TTA (~1 min)
Runs inference once, then sweeps the watershed thresholds over the cached
probabilities. The reported model reached 0.619 here.

In [ ]:
!python -m scripts.sweep_postprocess --ckpt runs/colab40/best.pt --tta

## 7. Prediction figure

In [ ]:
from IPython.display import Image, display

# image | ground truth | prediction for 2 fluorescence + 2 brightfield val images
!python -m scripts.make_figures predictions --ckpt runs/colab40/best.pt
display(Image('assets/predictions.png', width=800))

## 8. Download the figure
Add `assets/predictions.png` to the repo and uncomment the image line in the
README. The checkpoint stays in Drive (`MyDrive/nuclei/best.pt`, ~100 MB).

In [ ]:
from google.colab import files
files.download('assets/predictions.png')